In [1]:
%pip install z3 z3-solver

In [ ]:
from __future__ import annotations
import re
from z3 import *
from z3 import ExprRef, BoolRef  # type hints (from z3 import * không kéo các tên này)

class CompetitionSolver:
    def __init__(self):
        # Uninterpreted sort — Const(..., self.Object) cần SortRef từ DeclareSort, không dùng Sort('...')
        self.Object = DeclareSort("Object")
        self.predicates = {}
        self.constants = {}
        self.int_funcs = {}  # f: Object → Int (cho f(t) ≥ k, f(t) = k, ...)

    def _term(self, name: str, env: dict) -> ExprRef:
        if name in env:
            return env[name]
        if name not in self.constants:
            self.constants[name] = Const(name, self.Object)
        return self.constants[name]

    def _get_pred(self, name: str, arity: int):
        key = (name.strip(), arity)
        if key not in self.predicates:
            if arity == 1:
                self.predicates[key] = Function(name, self.Object, BoolSort())
            elif arity == 2:
                self.predicates[key] = Function(name, self.Object, self.Object, BoolSort())
            else:
                raise ValueError(f"Chưa hỗ trợ vị từ arity={arity}")
        return self.predicates[key]

    def _get_int_unary(self, name: str):
        """Hàm một ngôi Object → Int (dùng với so sánh số)."""
        name = name.strip()
        if name not in self.int_funcs:
            self.int_funcs[name] = Function(name, self.Object, IntSort())
        return self.int_funcs[name]

    @staticmethod
    def _split_top(s: str, sep: str) -> list[str]:
        """Tách theo sep chỉ ở độ sâu ngoặc () = 0 (tránh cắt nhầm ∧ bên trong ForAll(...))."""
        parts, buf, depth = [], [], 0
        for c in s:
            if c == "(":
                depth += 1
            elif c == ")":
                depth -= 1
            if c == sep and depth == 0:
                parts.append("".join(buf).strip())
                buf = []
            else:
                buf.append(c)
        parts.append("".join(buf).strip())
        return [p for p in parts if p]

    @staticmethod
    def _split_arrow_top(s: str) -> tuple[str, str | None]:
        depth = 0
        for i, c in enumerate(s):
            if c == "(":
                depth += 1
            elif c == ")":
                depth -= 1
            elif c == "→" and depth == 0:
                return s[:i].strip(), s[i + 1 :].strip()
        return s.strip(), None

    @staticmethod
    def _strip_outer_parens(s: str) -> str:
        s = s.strip()
        while s.startswith("(") and s.endswith(")"):
            depth = 0
            balanced = True
            for j, c in enumerate(s):
                if c == "(":
                    depth += 1
                elif c == ")":
                    depth -= 1
                    if depth == 0 and j < len(s) - 1:
                        balanced = False
                        break
            if balanced and depth == 0:
                s = s[1:-1].strip()
            else:
                break
        return s

    def parse_atom(self, atom_str: str, env: dict) -> BoolRef:
        atom_str = atom_str.strip()
        neg = atom_str.startswith("¬")
        if neg:
            atom_str = atom_str[1:].strip()
        # f(t) so sánh hằng Int: >=, <=, >, <, = (Unicode ≥ ≤ hoặc ASCII)
        m_cmp = re.match(
            r"^([a-zA-Z0-9_]+)\(\s*([a-zA-Z0-9_]+)\s*\)\s*(>=|<=|≥|≤|>|<|=)\s*(\d+)\s*$",
            atom_str,
        )
        if m_cmp:
            fname, arg, op, kstr = (
                m_cmp.group(1),
                m_cmp.group(2),
                m_cmp.group(3),
                m_cmp.group(4),
            )
            k = IntVal(int(kstr))
            t = self._get_int_unary(fname)(self._term(arg, env))
            if op in (">=", "≥"):
                expr = t >= k
            elif op in ("<=", "≤"):
                expr = t <= k
            elif op == ">":
                expr = t > k
            elif op == "<":
                expr = t < k
            elif op == "=":
                expr = t == k
            else:
                raise ValueError(f"Toán tử so sánh không hỗ trợ: {op}")
            return Not(expr) if neg else expr
        m = re.match(
            r"^([a-zA-Z0-9_]+)\(\s*([a-zA-Z0-9_]+)\s*(?:,\s*([a-zA-Z0-9_]+))?\s*\)\s*$",
            atom_str,
        )
        if not m:
            raise ValueError(f"Không parse được atom (vị từ / so sánh số): {atom_str}")
        pred, a1, a2 = m.group(1), m.group(2), m.group(3)
        if a2 is None:
            p = self._get_pred(pred, 1)
            expr = p(self._term(a1, env))
        else:
            p = self._get_pred(pred, 2)
            expr = p(self._term(a1, env), self._term(a2, env))
        return Not(expr) if neg else expr

    def parse_complex_side(self, side_str: str, env: dict) -> BoolRef:
        """∨ / ∧ tách trước (độ ưu tiên thấp hơn ¬); sau đó mới áp dụng ¬ tiền tố."""
        s = self._strip_outer_parens(side_str.strip())
        if "∨" in s:
            parts = self._split_top(s, "∨")
            if len(parts) > 1:
                xs = [self.parse_complex_side(p, env) for p in parts]
                return Or(*xs) if len(xs) > 1 else xs[0]
        if "∧" in s:
            parts = self._split_top(s, "∧")
            if len(parts) > 1:
                xs = [self.parse_complex_side(p, env) for p in parts]
                return And(*xs) if len(xs) > 1 else xs[0]
        if s.startswith("¬"):
            return Not(self.parse_complex_side(s[1:].strip(), env))
        return self.parse_atom(s, env)

    def string_to_z3_expr(self, fol_string: str) -> BoolRef:
        """Bóc ForAll(v,...); → / ∧ / ∨; vị từ Bool 1–2 ngôi; so sánh f(t) ≥ k / f(t) = k (f: Object→Int)."""
        s = fol_string.strip()
        qnames: list[str] = []
        while True:
            m = re.match(r"^\s*ForAll\(\s*(\w+)\s*,\s*", s, re.I)
            if not m:
                break
            qnames.append(m.group(1))
            s = s[m.end() :].strip()
        # Bỏ các ')' đóng lần lượt từng ForAll đã bóc ở cuối (vd. ...x))) -> ...x))
        for _ in qnames:
            if s.endswith(")"):
                s = s[:-1].strip()
        s = self._strip_outer_parens(s)
        env = {qn: Const(qn, self.Object) for qn in qnames}
        left, right = self._split_arrow_top(s)
        if right is None:
            final_expr = self.parse_complex_side(left, env)
        else:
            final_expr = Implies(
                self.parse_complex_side(left, env),
                self.parse_complex_side(right, env),
            )
        for qn in reversed(qnames):
            final_expr = ForAll(env[qn], final_expr)
        return final_expr

    def evaluate_quiz(self, premises_list, options_dict):
        base_solver = Solver()
        
        print("1. Đang nạp hệ thống Premises FOL của Qwen vào Solver...")
        for p_str in premises_list:
            z3_premise = self.string_to_z3_expr(p_str)
            base_solver.add(z3_premise)
            
        print("2. Kích hoạt thuật toán suy luận phản chứng trên các Option...")
        correct_option = None
        
        for label, opt_str in options_dict.items():
            test_solver = Solver()
            test_solver.add(base_solver.assertions())
            
            opt_z3 = self.string_to_z3_expr(opt_str)
            test_solver.add(Not(opt_z3)) # Phản chứng
            
            status = test_solver.check()
            print(f"   - Thử phương án {label}: Solver trả về -> {status}")
            if status == unsat:
                correct_option = label
                print(f"     => Phát hiện nghịch lý logic! Phương án {label} buộc phải ĐÚNG.")
                
        return correct_option

# =====================================================================
# CHẠY TEST ĐÃ KHẮC PHỤC LỖI
# =====================================================================
if __name__ == "__main__":
    qwen_output_premises = [
       "ForAll(x, (active_status(x) ∧ completed_courses(x) ≥ 5) → eligible_advanced(x))",
      "ForAll(x, eligible_advanced(x) → requires_approval(x))",
      "active_status(sarah)",
      "completed_courses(sarah) = 4",
      "has_approval(sarah)"
    ]

    # Options: dùng cùng tên vị từ / hàm Int như premises.
    qwen_output_options = {
    "A": "eligible_advanced(sarah) ∧ has_approval(sarah)",
    "B": "¬eligible_advanced(sarah) ∧ (completed_courses(sarah) < 5)",
    "C": "eligible_advanced(sarah) ∧ ¬has_approval(sarah)",
    "D": "active_status(sarah) → eligible_advanced(sarah)"
}

    engine = CompetitionSolver()
    answer = engine.evaluate_quiz(qwen_output_premises, qwen_output_options)
    
    print("\n" + "="*50)
    print(f"KẾT QUẢ CUỘC THI -> ĐÁP ÁN ĐÚNG LÀ: [ {answer} ]")
    print("="*50)

ImportError: cannot import name 'ExprRef' from 'z3' (/usr/local/lib/python3.12/dist-packages/z3/__init__.py)

In [ ]:
# --- One-shot prompt: NL premises → danh sách chuỗi FOL (khớp CompetitionSolver trong notebook này) ---
FOL_SYSTEM = """You translate natural-language PREMISES into first-order style strings for a Z3 pipeline.

OUTPUT RULES (strict):
- Output ONLY a Python list of strings, one formula per string. No markdown fences, no explanation before/after the list.
- Connectives (Unicode only): ∧  ∨  ¬  →  (implication is U+2192 →).
- Quantifiers: nested ASCII form, e.g. ForAll(x, ForAll(y, ...)) or ForAll(x, ...). Comma after the bound variable.
- Boolean atoms: name(term) or name(t1, t2). Identifiers: snake_case letters/digits/underscore.
- Integer unary functions Object→Int may appear ONLY as func(term) COMPARED to a non-negative literal:
  use >= or Unicode ≥ , <= or ≤ , > , < , = . Examples: completed_courses(sarah) = 4 , completed_courses(x) ≥ 5 .
- Do NOT output: ∃, types, arithmetic expressions, set notation, or English outside the list.

CONSISTENCY:
- Reuse the SAME symbols in every line; never introduce a synonym predicate for the same concept (e.g. do not mix can_take_advanced and eligible_advanced unless the text defines both).

OPTIONAL (only if the user explicitly asks for "closed definition" or "solver-ready iff"):
- You may add one line defining eligibility with ↔ using two implications: (φ → ψ) ∧ (ψ → φ) with →.

ONE-SHOT
NL premises:
\"\"\"
Any active student who completed at least five courses may register for advanced classes.
Anyone registered for advanced classes must have administrative approval.
Sarah is active. Sarah has completed exactly four courses. Sarah has administrative approval.
\"\"\"

Gold output (style to mimic):
[
  "ForAll(x, (active_status(x) ∧ completed_courses(x) ≥ 5) → eligible_advanced(x))",
  "ForAll(x, eligible_advanced(x) → requires_approval(x))",
  "active_status(sarah)",
  "completed_courses(sarah) = 4",
  "has_approval(sarah)"
]
"""


def build_fol_user_prompt(nl_premises: str, extra: str = "") -> str:
    """extra: ví dụ 'Thêm iff đủ-cần cho eligible_advanced nếu thiếu.' """
    parts = [
        "Convert the following NL text into the FOL list format from the system message.",
        "",
        "NL PREMISES:",
        '"""',
        nl_premises.strip(),
        '"""',
    ]
    if extra.strip():
        parts += ["", "ADDITIONAL INSTRUCTION:", extra.strip()]
    parts += ["", "Output ONLY the Python list of strings."]
    return "\n".join(parts)


# Ví dụ ghép với tokenizer (điền model/tokenizer của bạn):
# messages = [
#     {"role": "system", "content": FOL_SYSTEM},
#     {"role": "user", "content": build_fol_user_prompt(nl_text)},
# ]
# text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "Qwen/Qwen2.5-3B-Instruct"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained(model_name)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [7]:
# =============================================================================
# Luồng đầy đủ (1 cell): NL mẫu → LLM → JSON (premises + options) → Z3 → in kết quả
# Điều kiện: đã chạy cell định nghĩa CompetitionSolver + cell load model/tokenizer.
# =============================================================================
from __future__ import annotations

import json
import re

try:
    CompetitionSolver  # noqa: F821
except NameError as e:
    raise RuntimeError(
        "Chạy cell class CompetitionSolver + load model trước. Trên Colab: Kernel → Restart, rồi Run All (tránh RAM giữ code cũ)."
    ) from e

# --- Giống mục trong Logic_Based_Educational_Queries.json (stem MCQ + câu hỏi phụ) ---
SAMPLE_NL = """Based on the requirements, which statement about Sarah is correct?
A. She can take advanced classes because she has approval
B. She cannot take advanced classes due to insufficient completed courses
C. She is eligible but lacks approval
D. Her active status alone qualifies her
Does Sarah meet all requirements to take advanced classes, according to the premises?"""

FOL_QUIZ_SYSTEM = """You convert natural-language quiz context into FOL for a Z3 solver.

OUTPUT: ONLY one JSON object (no markdown, no text before/after) with exactly two keys:
- "premises": array of strings — universal facts / rules (ForAll, implications, ground facts).
- CRITICAL: "premises" MUST be a JSON array of STRINGS only. Correct: "premises":["ForAll(x, P(x))"]. Wrong: "premises":[{"ForAll(x, P(x))"}] — never wrap a formula in { } inside the array.
- "options": object with keys "A","B","C","D" — each value one formula string about the same constants (e.g. sarah).

SYNTAX (must match the solver):
- Connectives Unicode: ∧ ∨ ¬ → (arrow is → U+2192).
- Quantifiers: ForAll(x, formula) with comma after variable; nest as ForAll(x, ForAll(y, ...)).
- Boolean predicates: name(term) or name(t1, t2). snake_case identifiers.
- Do not use ↔; if you need iff, output one ForAll with ((φ → ψ) ∧ (ψ → φ)) using only → and ∧.
- Integer function Object→Int only as: func(term) >= n or ≥ n, <= ≤ > < = n (n non-negative digits). Example: completed_courses(sarah) = 4
- Use ONE vocabulary: if "advanced classes" maps to eligible_advanced(x), reuse it in options; do not invent synonym predicates.

If the user text does NOT state explicit premises, infer minimal standard policy premises that make the quiz well-posed (active status, completed course count vs threshold, eligibility, approval), then formalize A–D using the same symbols.

ONE-SHOT (style only; do not copy formulas blindly into new tasks):
NL: "Active students with at least 5 completed courses are eligible for advanced classes. Eligible students need approval. Sarah is active, completed 4 courses, has approval."
JSON:
{"premises":["ForAll(x, (active_status(x) ∧ completed_courses(x) ≥ 5) → eligible_advanced(x))","ForAll(x, eligible_advanced(x) → requires_approval(x))","active_status(sarah)","completed_courses(sarah) = 4","has_approval(sarah)","ForAll(x, ((active_status(x) ∧ completed_courses(x) ≥ 5) → eligible_advanced(x)) ∧ (eligible_advanced(x) → (active_status(x) ∧ completed_courses(x) ≥ 5)))"],"options":{"A":"eligible_advanced(sarah) ∧ has_approval(sarah)","B":"¬eligible_advanced(sarah) ∧ completed_courses(sarah) < 5","C":"eligible_advanced(sarah) ∧ ¬has_approval(sarah)","D":"active_status(sarah) → eligible_advanced(sarah)"}}
"""


def repair_llm_json_blob(blob: str) -> str:
    """Sửa JSON LLM: premise dạng { \"...\" } thay vì \"...\", smart quotes, trailing comma."""
    blob = blob.strip()
    blob = blob.replace("“", '"').replace("”", '"')
    blob = blob.replace("‘", "'").replace("’", "'")
    blob = re.sub(r",\s*([}\]])", r"\1", blob)
    pattern = re.compile(r'\{\s*"([^"]*)"\s*\}')
    prev = None
    while prev != blob:
        prev = blob
        blob = pattern.sub(lambda m: json.dumps(m.group(1)), blob)
    return blob


def extract_json_object(text: str) -> dict:
    text = text.strip()
    fm = re.search(r"```(?:json)?\s*", text, re.I)
    if fm:
        st = fm.end()
        en = text.find("```", st)
        if en >= 0:
            text = text[st:en].strip()
        else:
            m = re.search(r"```(?:json)?\s*(\{.*\})\s*```", text, re.DOTALL)
            if m:
                text = m.group(1).strip()
    start = text.find("{")
    if start < 0:
        raise ValueError("Không thấy '{' trong phản hồi model.")
    depth = 0
    for i in range(start, len(text)):
        if text[i] == "{":
            depth += 1
        elif text[i] == "}":
            depth -= 1
            if depth == 0:
                blob = text[start : i + 1]
                blob = repair_llm_json_blob(blob)
                try:
                    return json.loads(blob)
                except json.JSONDecodeError as e:
                    sn = blob[max(0, e.pos - 50) : e.pos + 50]
                    raise ValueError(
                        f"Lỗi JSON sau repair tại ~{e.pos}: {e.msg}. Đoạn: {sn!r}"
                    ) from e
    raise ValueError("JSON không cân ngoặc nhọn.")


def run_nl_to_fol_quiz(
    nl_text: str,
    *,
    max_new_tokens: int = 1024,
    temperature: float = 0.1,
) -> None:
    user = (
        "Convert the following text into the JSON format described in the system message.\n\n"
        "TEXT:\n"
        + json.dumps(nl_text, ensure_ascii=False)
        + "\n\nOutput ONLY the JSON object."
    )
    messages = [
        {"role": "system", "content": FOL_QUIZ_SYSTEM},
        {"role": "user", "content": user},
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)
    gen_kw = {"max_new_tokens": max_new_tokens}
    if temperature > 0:
        gen_kw["do_sample"] = True
        gen_kw["temperature"] = temperature
    else:
        gen_kw["do_sample"] = False
    gen = model.generate(**model_inputs, **gen_kw)
    gen_ids = gen[:, model_inputs.input_ids.shape[1] :]
    response = tokenizer.batch_decode(gen_ids, skip_special_tokens=True)[0]

    print("--- RAW MODEL ---")
    print(response)
    print("--- END RAW ---\n")

    data = extract_json_object(response)
    premises = data["premises"]
    options = data["options"]
    if not isinstance(premises, list) or not isinstance(options, dict):
        raise TypeError('JSON phải có "premises": list và "options": dict')

    print("--- PARSED PREMISES ---")
    for i, s in enumerate(premises, 1):
        print(f"  {i}. {s}")
    print("--- PARSED OPTIONS ---")
    for k in sorted(options.keys()):
        print(f"  {k}: {options[k]}")

    engine = CompetitionSolver()
    answer = engine.evaluate_quiz(premises, options)
    print("\n" + "=" * 50)
    print(f"ĐÁP ÁN SUY RA (entailment): [ {answer} ]")
    print("=" * 50)


run_nl_to_fol_quiz(SAMPLE_NL)


RuntimeError: Chạy cell class CompetitionSolver + load model trước. Trên Colab: Kernel → Restart, rồi Run All (tránh RAM giữ code cũ).